# VisionBridge — Base Model Training (Colab)

Clean real-data training workflow. Rebuilds a collision-safe ISL-CSLTR subset, validates feature/CTC contracts, runs a **semantic** real-data overfit gate, trains the existing Pose+Face Transformer, verifies non-trivial predictions on deterministic train/validation samples, then optionally pushes only the validated checkpoint + vocabulary.

A passing loss reduction or a single non-blank token is **not** sufficient. Space-only or blank-only collapse must fail the gate.

In [ ]:
import os, sys, subprocess, shutil, glob, hashlib, re, random, csv, textwrap, json
from pathlib import Path
import torch
REPO = Path('/content/VisionBridge')
if not (REPO / 'README.md').exists():
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO)], check=True)
else:
    status = subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'], text=True).strip()
    if status:
        raise RuntimeError('VisionBridge working tree is not clean. Refusing to overwrite local changes.')
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
BACKEND = REPO / 'backend'
sys.path.insert(0, str(BACKEND))
os.chdir(REPO)
print('HEAD:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip())
if not torch.cuda.is_available():
    raise RuntimeError('GPU required. In Colab select Runtime -> Change runtime type -> T4 GPU (or better).')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Isolated Python 3.12 MediaPipe runtime. The repository model/training code stays on the Colab Python runtime.
MP_ENV = Path('/content/visionbridge_mp312')
MP_PY = MP_ENV / 'bin/python'
MPL_CONFIG = Path('/content/visionbridge_mplconfig')
MPL_CONFIG.mkdir(parents=True, exist_ok=True)
uv = shutil.which('uv') or '/usr/local/bin/uv'
if not Path(uv).exists():
    raise RuntimeError('uv was not found in this Colab runtime.')
if not MP_ENV.exists():
    subprocess.run([uv,'python','install','3.12'], check=True)
    subprocess.run([uv,'venv','--python','3.12',str(MP_ENV)], check=True)
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['MPLCONFIGDIR'] = str(MPL_CONFIG)
probe = subprocess.run([str(MP_PY),'-c','import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)'], text=True, capture_output=True, env=env)
if probe.returncode != 0 or probe.stdout.strip() != '0.10.21':
    subprocess.run([uv,'pip','install','--python',str(MP_PY),'mediapipe==0.10.21','numpy==1.26.4','opencv-python-headless','pandas','matplotlib'], check=True, env=env)
version = subprocess.check_output([str(MP_PY),'-c','import mediapipe; print(mediapipe.__version__)'], text=True, env=env).strip()
if version != '0.10.21':
    raise RuntimeError(f'Expected MediaPipe 0.10.21, got {version!r}')
print('MediaPipe:', version)


In [ ]:
# Download the real ISL-CSLTR dataset and rebuild processed features from scratch.
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','kagglehub'], check=True)
    import kagglehub
root = Path(kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset'))
sentence_dirs = [Path(p) for p in glob.glob(str(root/'**'/'*Sentence_Level*'), recursive=True) if Path(p).is_dir() and 'Video' in Path(p).name]
if len(sentence_dirs) != 1:
    raise RuntimeError(f'Expected exactly one sentence-level video directory, found: {sentence_dirs}')
VIDEO_ROOT = sentence_dirs[0]
extensions = ('*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV','*.mkv','*.MKV')
files = []
for ext in extensions:
    files.extend(Path(p) for p in glob.glob(str(VIDEO_ROOT/'**'/ext), recursive=True))
files = sorted(set(files))
by_label = {}
for path in files:
    label = path.parent.name.replace('_',' ').strip()
    if label:
        by_label.setdefault(label, []).append(path)
rng = random.Random(42)
labels = sorted(by_label)
rng.shuffle(labels)
MAX_CLIPS = min(600, len(files))
MAX_CLIPS_PER_LABEL = 3
selected = []
for label in labels:
    clips = list(by_label[label])
    rng.shuffle(clips)
    for path in clips[:MAX_CLIPS_PER_LABEL]:
        selected.append((path, label))
    if len(selected) >= MAX_CLIPS:
        selected = selected[:MAX_CLIPS]
        break
print('Total videos:', len(files))
print('Selected videos:', len(selected))
print('Selected labels:', len({label for _, label in selected}))

DATA = REPO / 'data/processed/isltranslate'
shutil.rmtree(DATA, ignore_errors=True)
(DATA/'pose').mkdir(parents=True, exist_ok=True)
(DATA/'face').mkdir(parents=True, exist_ok=True)

def uid_for(path, label):
    relative = path.relative_to(VIDEO_ROOT).as_posix()
    safe_label = re.sub(r'[^a-z0-9]+','_',label.lower()).strip('_')[:40] or 'sample'
    safe_stem = re.sub(r'[^a-zA-Z0-9]+','_',path.stem).strip('_') or 'clip'
    digest = hashlib.sha1(relative.encode('utf-8')).hexdigest()[:10]
    return f'{safe_label}__{safe_stem}__{digest}'

# Keep helper outside the git repository so Colab support files cannot become accidental commits.
HELPER = Path('/content/visionbridge_extract_train_one.py')
HELPER.write_text(textwrap.dedent('''
    from pathlib import Path
    import sys, numpy as np
    from mediapipe.python.solutions import holistic
    repo = Path(sys.argv[1])
    sys.path.insert(0, str(repo / 'backend'))
    from scripts.extract_keypoints import extract_clip_keypoints
    with holistic.Holistic(static_image_mode=False, model_complexity=1) as h:
        pose, face = extract_clip_keypoints(sys.argv[2], h)
    assert pose.ndim == 2 and pose.shape[1] == 132
    assert face.ndim == 2 and face.shape[1] == 1404
    assert pose.shape[0] == face.shape[0] > 0
    assert np.isfinite(pose).all() and np.isfinite(face).all()
    np.save(sys.argv[3], pose)
    np.save(sys.argv[4], face)
'''), encoding='utf-8')

rows = []
seen = set()
failed = []
for number, (video, label) in enumerate(selected, start=1):
    uid = uid_for(video, label)
    if uid in seen:
        raise RuntimeError(f'UID collision: {uid}')
    seen.add(uid)
    pose_path = DATA/'pose'/f'{uid}.npy'
    face_path = DATA/'face'/f'{uid}.npy'
    result = subprocess.run([str(MP_PY), str(HELPER), str(REPO), str(video), str(pose_path), str(face_path)], text=True, capture_output=True, env=env)
    if result.returncode != 0:
        failed.append((str(video), result.stderr[-1000:]))
        continue
    rows.append({'uid': uid, 'text': label})
    if number % 25 == 0 or number == len(selected):
        print(f'{number}/{len(selected)} extracted; valid={len(rows)} failed={len(failed)}')
if len(rows) < 100:
    raise RuntimeError(f'Only {len(rows)} valid clips were extracted; at least 100 are required.')
with (DATA/'ISLTranslate.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['uid','text'])
    writer.writeheader()
    writer.writerows(rows)
print('Processed:', len(rows), 'Failed:', len(failed))
if failed:
    (DATA/'extraction_failures.txt').write_text('\n\n'.join(f'{video}\n{error}' for video, error in failed), encoding='utf-8')


In [ ]:
# Dataset contract + label sanity.
from app.training.isltranslate import ISLTranslateKeypointDataset, SimpleCharTokenizer
tok = SimpleCharTokenizer()
ds = ISLTranslateKeypointDataset(DATA, tokenizer=tok)
print('Dataset examples:', len(ds))
print('Vocabulary:', tok.vocab_size)
assert len(ds.examples) == len({example.uid for example in ds.examples})
for index in range(min(25, len(ds))):
    item = ds[index]
    assert item['pose'].ndim == 2 and item['pose'].shape[1] == 132
    assert item['face'].ndim == 2 and item['face'].shape[1] == 1404
    assert item['pose'].shape[0] == item['face'].shape[0] > 0
    assert item['labels'].numel() > 0
print('DATASET INTEGRITY: PASS')
print('Sample labels:')
for index in range(min(10, len(ds))):
    print(f'  {index}: {ds.examples[index].text!r}')


In [ ]:
# SEMANTIC real-data CTC sanity. This must reject blank/space/trivial-token collapse.
gate = subprocess.run([
    sys.executable, '-m', 'app.training.overfit_sanity',
    '--data-dir', str(DATA),
    '--samples', '1',
    '--steps', '2000',
    '--lr', '0.002',
    '--max-space-ratio', '0.90',
    '--min-meaningful-unique', '2',
    '--max-mean-cer', '0.90',
], cwd=REPO, env={**os.environ, 'PYTHONPATH': str(BACKEND)}, text=True, capture_output=True)
print(gate.stdout)
if gate.stderr:
    print('STDERR:\n', gate.stderr)
if gate.returncode != 0:
    raise RuntimeError('SEMANTIC OVERFIT GATE FAILED. Full training is blocked.')
print('SEMANTIC OVERFIT GATE: PASS')


In [ ]:
# Full training. The training script independently checks trainability/gradients and removes stale output when not resuming.
OUT = REPO / 'backend/app/models/weights/base_model.pt'
CKPT_DIR = Path('/content/visionbridge_training_ckpt')
shutil.rmtree(CKPT_DIR, ignore_errors=True)
cmd = [
    sys.executable, '-m', 'app.training.train_base_model',
    '--data-dir', str(DATA),
    '--output', str(OUT),
    '--epochs', '20',
    '--batch-size', '2',
    '--lr', '3e-4',
    '--weight-decay', '1e-2',
    '--seed', '42',
    '--num-workers', '2',
    '--checkpoint-dir', str(CKPT_DIR),
]
train_run = subprocess.run(cmd, cwd=REPO, env={**os.environ, 'PYTHONPATH': str(BACKEND)}, text=True)
if train_run.returncode != 0:
    raise RuntimeError('Full training failed.')
vocab_path = OUT.with_suffix('.vocab.json')
if not OUT.exists() or not vocab_path.exists():
    raise RuntimeError('Training completed without both checkpoint and vocabulary.')
print('TRAINING: PASS')
print('Checkpoint:', OUT)
print('Vocabulary:', vocab_path)


In [ ]:
# Strong post-training semantic acceptance. The old non-blank-only gate is intentionally gone.
import torch
from torch.utils.data import random_split
from app.models.base_model import load_frozen_base_model
from app.services import inference_service
from app.training.isltranslate import _downsample_to_max_length

tokenizer = SimpleCharTokenizer.load(vocab_path)
val_n = max(1, int(len(ds) * 0.1))
train_subset, val_subset = random_split(ds, [len(ds)-val_n, val_n], generator=torch.Generator().manual_seed(42))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_frozen_base_model(str(OUT), vocab_size=tokenizer.vocab_size).to(device).eval()
inference_service._id_to_token = inference_service._load_vocab(str(OUT))
if len(inference_service._id_to_token) != model.output_head.out_features:
    raise RuntimeError('Checkpoint/vocabulary mismatch.')

def levenshtein(a, b):
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1]+1, previous[j]+1, previous[j-1]+(ca != cb)))
        previous = current
    return previous[-1]

def semantic_check(index, tag):
    item = ds[index]
    pose, face = _downsample_to_max_length(item['pose'], item['face'], item['uid'])
    pose = pose.unsqueeze(0).to(device)
    face = face.unsqueeze(0).to(device)
    lengths = torch.tensor([pose.shape[1]], dtype=torch.long, device=device)
    with torch.inference_mode():
        logits = model(pose, face, lengths)
    if not torch.isfinite(logits).all():
        raise RuntimeError(f'{tag}: non-finite logits')
    probs = torch.softmax(logits[0], dim=-1)
    ids = probs.argmax(dim=-1).cpu().tolist()
    blank_id = tokenizer.token_to_id['<blank>']
    space_id = tokenizer.token_to_id[' ']
    space_ratio = sum(x == space_id for x in ids) / len(ids)
    meaningful_ids = [x for x in ids if x != blank_id and tokenizer.id_to_token[x].strip()]
    collapsed = []
    previous = None
    for x in ids:
        if x != previous:
            collapsed.append(x)
        previous = x
    collapsed = [x for x in collapsed if x != blank_id]
    prediction = ''.join(tokenizer.id_to_token[x] for x in collapsed)
    target = item['text']
    char_error = levenshtein(prediction.lower(), target.lower()) / max(len(target), 1)
    unique_meaningful = len(set(meaningful_ids))
    print(f'{tag}: truth={target!r} prediction={prediction!r} CER={char_error:.3f} space_ratio={space_ratio:.3f} unique_meaningful={unique_meaningful}')
    if not prediction.strip():
        raise RuntimeError(f'{tag}: empty/whitespace-only prediction')
    if space_ratio >= 0.90:
        raise RuntimeError(f'{tag}: space-token collapse')
    if unique_meaningful < 2:
        raise RuntimeError(f'{tag}: fewer than two meaningful predicted tokens')
    return prediction, char_error

train_results = []
val_results = []
for subset, results, tag in [(train_subset, train_results, 'TRAIN'), (val_subset, val_results, 'VAL')]:
    for index in subset.indices[:4]:
        results.append(semantic_check(index, tag))

train_mean_cer = sum(x[1] for x in train_results) / len(train_results)
val_mean_cer = sum(x[1] for x in val_results) / len(val_results)
print()
print('=' * 72)
print(f'TRAIN mean CER: {train_mean_cer:.4f}')
print(f'VAL mean CER:   {val_mean_cer:.4f}')
print('=' * 72)
if train_mean_cer > 0.95:
    raise RuntimeError(f'MODEL ACCEPTANCE FAILED: train mean CER {train_mean_cer:.4f} is too high.')
if val_mean_cer > 1.00:
    raise RuntimeError(f'MODEL ACCEPTANCE FAILED: validation mean CER {val_mean_cer:.4f} is too high.')
if len(set(pred for pred, _ in train_results + val_results)) < 2:
    raise RuntimeError('MODEL ACCEPTANCE FAILED: all checked samples produce the same prediction.')
print('MODEL ACCEPTANCE: PASS')


In [ ]:
# Push ONLY after semantic acceptance passes. No destructive git reset/clean operations.
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = os.environ.get('GITHUB_TOKEN')
if not token:
    print('GITHUB_TOKEN not configured. Checkpoint remains local.')
else:
    subprocess.run(['git','-C',str(REPO),'config','user.name','VisionBridge Trainer'], check=True)
    subprocess.run(['git','-C',str(REPO),'config','user.email','visionbridge-trainer@users.noreply.github.com'], check=True)
    remote = 'https://x-access-token:' + token + '@github.com/BharathWaj-K-R/VisionBridge.git'
    subprocess.run(['git','-C',str(REPO),'remote','set-url','origin',remote], check=True)
    allowed = {'backend/app/models/weights/base_model.pt','backend/app/models/weights/base_model.vocab.json'}
    subprocess.run(['git','-C',str(REPO),'add','--',*sorted(allowed)], check=True)
    staged = subprocess.check_output(['git','-C',str(REPO),'diff','--cached','--name-only'], text=True).splitlines()
    unexpected = [path for path in staged if path not in allowed]
    if unexpected:
        raise RuntimeError('Unexpected staged files: ' + repr(unexpected))
    if not staged:
        raise RuntimeError('Nothing to push; checkpoint/vocabulary are unchanged.')
    subprocess.run(['git','-C',str(REPO),'commit','-m','train: push semantically validated base model'], check=True)
    subprocess.run(['git','-C',str(REPO),'push','origin','main'], check=True)
    sha = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
    print('MODEL PUSH: PASS')
    print('Commit:', sha)
    print('Tracked remaining changes:')
    print(subprocess.check_output(['git','-C',str(REPO),'status','--short'], text=True))
